In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import gym
import numpy as np

# تنظیمات دستگاه
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# شبکه Actor-Critic
class ActorCritic(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(ActorCritic, self).__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh()
        )
        self.actor = nn.Linear(hidden_dim, output_dim)
        self.critic = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        shared_out = self.shared(x)
        logits = self.actor(shared_out)
        value = self.critic(shared_out)
        probs = torch.softmax(logits, dim=-1)
        return probs, value

# محیط
env = gym.make('CartPole-v1')

# پارامترها
input_dim = env.observation_space.shape[0]  # 4
output_dim = env.action_space.n             # 2
hidden_dim = 64
lr = 3e-4
gamma = 0.99
eps_clip = 0.2
K_epochs = 4  # تعداد دفعات به‌روزرسانی روی داده‌های یک batch
batch_size = 2048  # حداکثر تعداد گام‌ها در یک batch
num_episodes = 500

# شبکه و بهینه‌ساز
model = ActorCritic(input_dim, hidden_dim, output_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)

# ذخیره‌سازی تراژکتوری
states = []
actions = []
log_probs = []
rewards = []
dones = []
values = []

def compute_advantages(rewards, values, dones, gamma=0.99, lam=0.95):
    """
    محاسبه Advantage با GAE (Generalized Advantage Estimation)
    """
    advantages = []
    gae = 0
    next_value = 0
    for i in reversed(range(len(rewards))):
        if dones[i]:
            td_error = rewards[i] - values[i]
        else:
            td_error = rewards[i] + gamma * values[i+1] - values[i]
        gae = td_error + gamma * lam * (1 - dones[i]) * gae
        advantages.insert(0, gae)
    return torch.tensor(advantages, dtype=torch.float32).to(device)

# آموزش PPO
for episode in range(num_episodes):
    state = env.reset()[0]
    total_reward = 0

    while True:
        # Forward pass
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            probs, value = model(state_tensor)
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()
            log_prob = dist.log_prob(action)
            value = value.squeeze()

        # انجام عمل
        next_state, reward, terminated, truncated, _ = env.step(action.item())
        done = terminated or truncated
        total_reward += reward

        # ذخیره داده
        states.append(state)
        actions.append(action.item())
        log_probs.append(log_prob.item())
        rewards.append(reward)
        dones.append(done)
        values.append(value.item())

        state = next_state

        # اگر batch پر شد یا episode تمام شد
        if len(states) >= batch_size or done:
            # تبدیل به تانسور
            states_tensor = torch.FloatTensor(states).to(device)
            actions_tensor = torch.LongTensor(actions).to(device)
            old_log_probs = torch.FloatTensor(log_probs).to(device)
            returns = []
            R = 0
            for r, done in zip(reversed(rewards), reversed(dones)):
                R = r + gamma * R * (1 - done)
                returns.insert(0, R)
            returns = torch.tensor(returns, dtype=torch.float32).to(device)

            # محاسبه Advantage
            values_tensor = torch.FloatTensor(values).to(device)
            advantages = returns - values_tensor
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

            # به‌روزرسانی چندین بار روی داده‌های جمع‌آوری‌شده
            for _ in range(K_epochs):
                probs, value = model(states_tensor)
                dist = torch.distributions.Categorical(probs)
                new_log_probs = dist.log_prob(actions_tensor)
                entropy = dist.entropy().mean()

                # نسبت احتمال
                ratios = (new_log_probs - old_log_probs).exp()

                # Surrogate Loss
                surr1 = ratios * advantages
                surr2 = torch.clamp(ratios, 1 - eps_clip, 1 + eps_clip) * advantages
                actor_loss = -torch.min(surr1, surr2).mean()
                critic_loss = (returns - value.squeeze()).pow(2).mean()
                total_loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy

                optimizer.zero_grad()
                total_loss.backward()
                optimizer.step()

            # پاک‌کردن بافر
            states, actions, log_probs, rewards, dones, values = [], [], [], [], [], []

        if done:
            break

    if episode % 50 == 0:
        print(f"Episode {episode}, Total Reward: {total_reward}")

print("Training finished.")